# Notebook 05: Analise Espacial

**Projeto:** `migracao-venezuelana-oeste-sc`  
**Autores:** Leonardo Rafael Santos Leitao e Vicente Neves da Silva Ribeiro (UFFS)  
**Data:** Abril de 2026

---

## Objetivo

Este notebook realiza analise espacial da distribuicao da populacao venezuelana no Oeste de Santa Catarina, incluindo:

1. Mapa coropletico da proporcao de venezuelanos por municipio;
2. Analise de autocorrelacao espacial (Moran I);
3. Clusters locais (LISA).

> **Nota:** Dados simulados utilizados ate obtencao das malhas municipais do IBGE e dados reais.

## 1. Configuracao

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 10)
plt.rcParams["figure.dpi"] = 100

PROJECT_ROOT = Path(".").resolve().parent
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Tentativa de importar geopandas e pysal
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False
    print("[AVISO] geopandas nao instalado. Mapas nao serao gerados.")

try:
    from pysal.explore import esda
    from pysal.lib import weights
    HAS_PYSAL = True
except ImportError:
    HAS_PYSAL = False
    print("[AVISO] pysal nao instalado. Analise espacial desabilitada.")

## 2. Dados Simulados

In [ ]:
np.random.seed(42)
municipios = [
    {"nome": "Chapeco", "codigo": 4204202, "pop_total": 224013, "lat": -27.100, "lon": -52.615},
    {"nome": "Xanxere", "codigo": 4219407, "pop_total": 52133, "lat": -26.873, "lon": -52.404},
    {"nome": "Concordia", "codigo": 4204301, "pop_total": 81251, "lat": -27.234, "lon": -52.027},
    {"nome": "Joacaba", "codigo": 4209003, "pop_total": 30171, "lat": -27.178, "lon": -51.505},
    {"nome": "Sao Miguel do Oeste", "codigo": 4217204, "pop_total": 44952, "lat": -26.725, "lon": -53.518},
]

for m in municipios:
    m["pop_venezuelana"] = int(m["pop_total"] * np.random.uniform(0.001, 0.015))
    m["pct_venezuelana"] = (m["pop_venezuelana"] / m["pop_total"]) * 100

df_map = pd.DataFrame(municipios)
display(df_map[["nome", "pop_total", "pop_venezuelana", "pct_venezuelana"]])

## 3. Mapa Coropletico (Placeholder sem GeoJSON)

In [ ]:
# Sem geopandas, geramos um scatter plot geografico como placeholder
fig, ax = plt.subplots(figsize=(10, 10))
scatter = ax.scatter(
    df_map["lon"], df_map["lat"],
    s=df_map["pop_venezuelana"] * 5,
    c=df_map["pct_venezuelana"],
    cmap="YlOrRd",
    alpha=0.8,
    edgecolors="black",
)
for _, row in df_map.iterrows():
    ax.annotate(row["nome"], (row["lon"], row["lat"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
ax.set_title("Proporcao de Venezuelanos por Municipio — Oeste de SC (Placeholder)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
cbar = plt.colorbar(scatter, ax=ax, shrink=0.6)
cbar.set_label("% Venezuelanos")
sns.despine()
plt.tight_layout()
fig_path = FIGURES_DIR / "05_mapa_espacial_placeholder.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 4. Moran I e LISA (Placeholder)

In [ ]:
if HAS_PYSAL:
    # Placeholder: construcao de matriz de pesos a partir de distancias
    coords = df_map[["lon", "lat"]].values
    w = weights.distance.KNN.from_array(coords, k=2)
    w.transform = "r"
    
    moran = esda.Moran(df_map["pct_venezuelana"], w)
    print(f"Moran I: {moran.I:.4f}")
    print(f"p-valor (simulado): {moran.p_sim:.4f}")
    
    lisa = esda.Moran_Local(df_map["pct_venezuelana"], w)
    df_map["lisa_cluster"] = lisa.q
    print("\nClusters LISA (1=HH, 2=LH, 3=LL, 4=HL):")
    print(df_map[["nome", "pct_venezuelana", "lisa_cluster"]])
else:
    print("[INFO] pysal nao disponivel. Analise Moran I e LISA serao executadas apos instalacao.")

## Proximos Passos

- [ ] Baixar malhas municipais do IBGE (shapefile/GeoJSON);
- [ ] Instalar `geopandas` e `pysal`;
- [ ] Calcular Moran I e LISA com dados reais;
- [ ] Gerar mapa coropletico oficial com geopandas.